In [1]:
!apt-get update -qq
!apt-get install -y zstd

W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)



The following NEW packages will be installed:
  zstd
0 upgraded, 1 newly installed, 0 to remove and 224 not upgraded.
Need to get 603 kB of archives.
After this operation, 1,695 kB of additional disk space will be used.
Get:1 http://archive.ubuntu.com/ubuntu jammy/main amd64 zstd amd64 1.4.8+dfsg-3build1 [603 kB]
Fetched 603 kB in 0s (10.7 MB/s)
Selecting previously unselected package zstd.
(Reading database ... 124626 files and directories currently installed.)
Preparing to unpack .../zstd_1.4.8+dfsg-3build1_amd64.deb ...
Unpacking zstd (1.4.8+dfsg-3build1) ...
Setting up zstd (1.4.8+dfsg-3build1) ...
Processing triggers for man-db (2.10.2-1) ...


In [2]:
!curl -fsSL https://ollama.com/install.sh | sh

>>> Installing ollama to /usr/local
>>> Downloading ollama-linux-amd64.tar.zst
######################################################################## 100.0%
>>> Creating ollama user...
>>> Adding ollama user to video group...
>>> Adding current user to ollama group...
>>> Creating ollama systemd service...
>>> The Ollama API is now available at 127.0.0.1:11434.
>>> Install complete. Run "ollama" from the command line.


In [3]:
!which ollama

/usr/local/bin/ollama


In [4]:
import subprocess
import time

process = subprocess.Popen(
    ["ollama", "serve"],
    stdout=subprocess.DEVNULL,
    stderr=subprocess.DEVNULL
)

time.sleep(5)

print("Ollama started")

Ollama started


In [5]:
import requests

r = requests.get("http://localhost:11434/api/tags")

print(r.text)


{"models":[]}


In [6]:
import subprocess

subprocess.run(
    ["ollama", "pull", "qwen2.5:7b"]
)

pulling manifest ⠋ pulling manifest ⠙ pulling manifest ⠹ pulling manifest ⠸ pulling manifest ⠼ pulling manifest ⠴ pulling manifest ⠦ pulling manifest ⠧ pulling manifest ⠇ pulling manifest ⠏ pulling manifest 
pulling 2bada8a74506:   1% ▕                  ▏  34 MB/4.7 GB                  pulling manifest 
pulling 2bada8a74506:   2% ▕                  ▏  82 MB/4.7 GB                  pulling manifest 
pulling 2bada8a74506:   4% ▕                  ▏ 176 MB/4.7 GB                  pulling manifest 
pulling 2bada8a74506:   6% ▕█                 ▏ 273 MB/4.7 GB                  pulling manifest 
pulling 2bada8a74506:   7% ▕█                 ▏ 320 MB/4.7 GB                  pulling manifest 
pulling 2bada8a74506:   9% ▕█                 ▏ 418 MB/4.7 GB                  pulling manifest 
pulling 2bada8a74506:  11% ▕█                 ▏ 515 MB/4.7 GB                  pulling manifest 
pulling 2bada8a74506:  12% ▕██                ▏ 574 MB/4.7 GB                  pulling manifest 
pulling 2bada8a7

CompletedProcess(args=['ollama', 'pull', 'qwen2.5:7b'], returncode=0)

In [7]:
"""
KAGGLE-READY VIETNAMESE HAM DATA GENERATION PIPELINE
----------------------------------------------------
Features:
- Async Ollama generation
- SQLite checkpointing
- Fast semantic dedup
- Batch embedding & database insert
- Legitimate/Normal User personas & taxonomy
"""

import re
import csv
import time
import random
import sqlite3
import asyncio
import aiohttp
from collections import deque
import numpy as np
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity

# =========================================================
# CONFIG
# =========================================================
OLLAMA_URL = "http://localhost:11434/api/generate"
MODEL_NAME = "qwen2.5:7b"
TARGET_SAMPLES = 10000  # Khuyến nghị 10,000 để tạo tỷ lệ 1:1
CONCURRENCY_LIMIT = 2
BATCH_SIZE = 8
DB_FILE = "/kaggle/working/ham_dataset.sqlite"
CSV_EXPORT = "/kaggle/working/vietnamese_ham_dataset.csv"

EMBEDDING_MODEL = "paraphrase-multilingual-MiniLM-L12-v2"
SEMANTIC_THRESHOLD = 0.87
MAX_EMBEDDING_MEMORY = 50000

# =========================================================
# TAXONOMY (HAM / NORMAL USAGE)
# =========================================================
PLATFORMS = [
    "Facebook Comment (Bài đăng của bạn bè, group cộng đồng)",
    "Zalo Message (Nhắn tin với gia đình, đồng nghiệp)",
    "Tiktok Comment (Bình luận video giải trí, review đồ ăn)",
    "Youtube Comment (Bàn luận âm nhạc, vlog, hướng dẫn học tập)",
    "Shopee/Lazada Review (Đánh giá chất lượng thật sau khi mua)",
    "Forum Discussion (Voz, TinhTe - hỏi đáp công nghệ, chuyện đời sống)",
    "SMS (Nhắn tin cá nhân, báo bận, hẹn hò)"
]

PERSONAS = [
    "Dân văn phòng (chia sẻ áp lực công việc, rủ đi ăn trưa, than thở deadline)",
    "Sinh viên/Học sinh (hỏi bài, than vãn thi cử, bàn chuyện idol)",
    "Mẹ bỉm sữa bình thường (hỏi kinh nghiệm nuôi con, nấu ăn, mua sắm)",
    "Người mua hàng kỹ tính (review chi tiết khen/chê sản phẩm chân thực)",
    "Thanh niên mê công nghệ (hỏi đáp về build PC, review code, fix bug)",
    "Người yêu thích du lịch/ẩm thực (hỏi địa điểm, chia sẻ cảm nhận quán ăn)",
    "Người dùng mạng xã hội gen Z (dùng nhiều teencode, đùa cợt, bắt trend hài hước)"
]

TRENDING_KEYWORDS = [
    "deadline", "cuối tuần", "OT", "trà sữa", "mưa ngập", "lương", 
    "học phí", "pass môn", "review", "chất vải", "ship nhanh", 
    "ngon", "rẻ", "lỗi win", "fix bug", "đi phượt", "phim rạp"
]

TAXONOMY = {
    "casual_chat": {
        "topics": ["hỏi thăm sức khỏe", "rủ đi chơi/cafe", "than vãn thời tiết", "kể chuyện đi đường", "bàn luận phim ảnh"], 
        "hooks": ["nhớ nha", "chán ghê", "vui xỉu", "ai rảnh không", "thế là xong"]
    },
    "q_and_a": {
        "topics": ["hỏi chỗ sửa xe", "xin tư vấn mua điện thoại", "hỏi đường đi", "hỏi cách làm thủ tục hành chính", "nhờ giải bài tập"], 
        "hooks": ["ai biết chỉ với", "giúp em với ạ", "mọi người cho hỏi", "đang phân vân quá", "bác nào có kinh nghiệm"]
    },
    "genuine_review": {
        "topics": ["review quán ăn mới mở", "đánh giá quần áo vừa nhận", "nhận xét phim chiếu rạp", "chia sẻ trải nghiệm đi du lịch"], 
        "hooks": ["thất vọng thực sự", "đáng tiền nha", "phục vụ hơi chậm", "form đẹp như hình", "chắc chắn sẽ quay lại"]
    },
    "work_study": {
        "topics": ["chạy deadline", "áp lực công việc", "than thở sếp", "mừng pass môn", "tìm tài liệu ôn thi"], 
        "hooks": ["ngủ gục", "chưa xong nữa", "may quá qua môn", "đói meo", "xin file tài liệu với"]
    }
}

SEED_EXAMPLES = [
    "Mọi người cho em hỏi tầm 15 củ thì mua laptop nào code mượt, ít nóng máy ạ?",
    "Quán này đồ ăn khá ổn, không gian thoáng nhưng phục vụ lên món hơi chậm. Tạm cho 4 sao nhé.",
    "Nay sếp lại bắt OT đến 8h tối, chắc xỉu ngang ở cty quá anh em ơi :(",
    "Áo chất mát, form rộng mặc thoải mái. Shop giao hàng nhanh. Sẽ ủng hộ tiếp.",
    "Cuối tuần này có ai lập team đi coi Dune 2 không, nghe bảo kỹ xảo đỉnh lắm.",
    "Bác nào rành về xe cho em hỏi, xe Vision dạo này đi hay bị hụp ga là bệnh gì vậy ạ?",
    "Thôi nay bận rồi, hẹn chủ nhật tuần sau đi cafe nha bà.",
    "Em giải mãi bài toán vi tích phân này không ra, anh chị nào rảnh gợi ý em hướng làm với ạ."
]

# =========================================================
# DATABASE & DEDUP (Giữ nguyên cấu trúc)
# =========================================================
class Database:
    def __init__(self, db_path):
        self.conn = sqlite3.connect(db_path, check_same_thread=False)
        self.cursor = self.conn.cursor()
        self.cursor.execute("""
        CREATE TABLE IF NOT EXISTS dataset(
            id INTEGER PRIMARY KEY AUTOINCREMENT,
            text TEXT UNIQUE,
            label TEXT,
            category TEXT,
            style TEXT,
            source TEXT,
            platform TEXT,
            persona TEXT
        )
        """)
        self.conn.commit()
        self.buffer = []

    def insert(self, row):
        self.buffer.append((
            row["text"], row["label"], row["category"], 
            row["style"], row["source"], row["platform"], row["persona"]
        ))
        if len(self.buffer) >= 50:
            self.flush()

    def flush(self):
        if not self.buffer: return
        self.cursor.executemany("""
        INSERT OR IGNORE INTO dataset
        (text, label, category, style, source, platform, persona)
        VALUES (?, ?, ?, ?, ?, ?, ?)
        """, self.buffer)
        self.conn.commit()
        self.buffer.clear()

    def count(self):
        self.cursor.execute("SELECT COUNT(*) FROM dataset")
        return self.cursor.fetchone()[0]

    def get_all_texts(self):
        self.cursor.execute("SELECT text FROM dataset")
        return [x[0] for x in self.cursor.fetchall()]

    def export_csv(self, path):
        self.flush()
        self.cursor.execute("SELECT text, label, category, style, source, platform, persona FROM dataset")
        rows = self.cursor.fetchall()
        with open(path, "w", newline="", encoding="utf-8-sig") as f:
            writer = csv.writer(f)
            writer.writerow(["text", "label", "category", "style", "source", "platform", "persona"])
            writer.writerows(rows)

class SemanticDedup:
    def __init__(self, texts):
        print("Loading embedding model...")
        self.model = SentenceTransformer(EMBEDDING_MODEL)
        self.texts = list(texts)
        if texts:
            self.embeddings = self.model.encode(texts, batch_size=64, convert_to_numpy=True, show_progress_bar=True)
        else:
            self.embeddings = np.empty((0, 384))

    def is_duplicate(self, text):
        if len(self.embeddings) == 0: return False
        emb = self.model.encode([text], convert_to_numpy=True)
        recent_embs = self.embeddings[-2000:]
        scores = cosine_similarity(emb, recent_embs)[0]
        return scores.max() > SEMANTIC_THRESHOLD

    def add_batch(self, texts):
        if not texts: return
        embs = self.model.encode(texts, batch_size=32, convert_to_numpy=True)
        self.embeddings = np.vstack([self.embeddings, embs])
        self.texts.extend(texts)
        if len(self.embeddings) > MAX_EMBEDDING_MEMORY:
            self.embeddings = self.embeddings[-MAX_EMBEDDING_MEMORY:]
            self.texts = self.texts[-MAX_EMBEDDING_MEMORY:]

# =========================================================
# MUTATION ENGINE (Sửa nhẹ lại để phù hợp văn phong bình thường)
# =========================================================
ABBREVIATIONS = {
    "không": ["ko", "k", "khong"],
    "mình": ["mk", "mik", "m"],
    "được": ["dc", "đc"],
    "anh em": ["ae"],
    "mọi người": ["mn"],
    "bây giờ": ["giờ"],
    "điện thoại": ["đt", "phone"]
}

def apply_abbrev(text):
    for k, vals in ABBREVIATIONS.items():
        if k in text:
            text = text.replace(k, random.choice(vals), 1)
    return text

def apply_emoji(text):
    emojis = ["😂", "😭", "😍", "🤔", "👍", "🙏", "🥲", "😅"]
    return text + " " + random.choice(emojis)

def mutate_text(text):
    # Với Ham data, ta chủ yếu dùng viết tắt và emoji, bỏ các lỗi cố tình lặp chữ (nnn) của bot
    funcs = [apply_abbrev, apply_emoji]
    fn = random.choice(funcs)
    return fn(text)

# =========================================================
# VALIDATION
# =========================================================
REFUSAL_RE = re.compile(r"(tôi không thể|xin lỗi|tôi là AI|trợ lý ảo|vi phạm)", re.IGNORECASE)

def validate_text(text):
    text = text.strip()
    if len(text) < 10: return False  # Bình luận thường có thể ngắn hơn spam một chút
    if len(text) > 400: return False
    if REFUSAL_RE.search(text): return False
    if re.search(r"[\u4e00-\u9fff]", text): return False
    return True

# =========================================================
# PROMPT (SỬA ĐỂ LLM SINH DỮ LIỆU SẠCH)
# =========================================================
def build_prompt(category, platform, persona, recent_words):
    topic = random.choice(TAXONOMY[category]["topics"])
    keyword = random.choice(TRENDING_KEYWORDS)
    examples = "\n".join(["- " + x for x in random.sample(SEED_EXAMPLES, 2)])
    avoid = ""
    
    if recent_words:
        avoid = "Tuyệt đối không lặp lại các từ vựng sau: " + ", ".join(recent_words)
        
    prompt = f"""Bạn đang tạo dữ liệu giả lập cho nghiên cứu NLP phân loại văn bản.
Hãy viết {BATCH_SIZE} bình luận của NGƯỜI DÙNG THẬT (không phải spam, không quảng cáo lừa đảo, không điều hướng link) bằng tiếng Việt.

Bối cảnh:
- Nền tảng: {platform}
- Người viết (Persona): {persona}
- Chủ đề: {topic}
- Từ khóa nên có: {keyword}

Yêu cầu:
- Mỗi dòng là 1 comment độc lập.
- NỘI DUNG SẠCH, BÌNH THƯỜNG, ĐỜI THƯỜNG (Ham data).
- Không đánh số đầu dòng, không dùng markdown.
- Không giải thích.
- Văn phong tự nhiên như người Việt hay chat, có hỉ nộ ái ố, khen chê rõ ràng.
{avoid}

Ví dụ:
{examples}"""
    return prompt

# =========================================================
# OLLAMA & WORKER & MAIN
# =========================================================
async def fetch_llm(session, prompt):
    payload = {
        "model": MODEL_NAME,
        "prompt": prompt,
        "stream": False,
        "options": {
            "temperature": round(random.uniform(0.6, 0.9), 2),
            "top_p": 0.9,
            "repeat_penalty": 1.1,
        }
    }
    try:
        async with session.post(OLLAMA_URL, json=payload, timeout=120) as response:
            if response.status == 200:
                data = await response.json()
                return data.get("response", "")
    except Exception as e:
        print("LLM ERROR:", e)
    return ""

async def worker(session, db, dedup, sem, recent_words):
    async with sem:
        category = random.choice(list(TAXONOMY.keys()))
        platform = random.choice(PLATFORMS)
        persona = random.choice(PERSONAS)
        prompt = build_prompt(category, platform, persona, recent_words)
        raw = await fetch_llm(session, prompt)
        
        accepted = []
        accepted_texts = []
        
        for line in raw.split("\n"):
            line = line.strip()
            line = re.sub(r"^[-•*]\s*", "", line)
            
            if not validate_text(line):
                continue
                
            if random.random() < 0.4:
                line = mutate_text(line)
                
            if dedup.is_duplicate(line):
                continue
                
            accepted_texts.append(line)
            accepted.append({
                "text": line,
                "label": "ham",  # <-- Đã đổi label thành ham
                "category": category,
                "style": "generated",
                "source": "ollama",
                "platform": platform,
                "persona": persona,
            })
            
        dedup.add_batch(accepted_texts)
        for item in accepted:
            db.insert(item)
            
        for text in accepted_texts:
            words = [w for w in text.split() if len(w) > 4]
            if words:
                recent_words.append(random.choice(words))
                
        return len(accepted)

async def run_pipeline():
    print("Initializing database...")
    db = Database(DB_FILE)
    existing = db.get_all_texts()
    dedup = SemanticDedup(existing)
    current = db.count()
    
    print(f"Starting from {current}/{TARGET_SAMPLES}")
    recent_words = deque(maxlen=20)
    sem = asyncio.Semaphore(CONCURRENCY_LIMIT)
    
    async with aiohttp.ClientSession() as session:
        while current < TARGET_SAMPLES:
            tasks = []
            for _ in range(CONCURRENCY_LIMIT):
                tasks.append(worker(session, db, dedup, sem, recent_words))
            results = await asyncio.gather(*tasks)
            db.flush()
            current = db.count()
            print(f"+{sum(results)} | TOTAL: {current}/{TARGET_SAMPLES}")
            await asyncio.sleep(0.3)
            
    print("Exporting CSV...")
    db.export_csv(CSV_EXPORT)
    print("DONE")

if __name__ == "__main__":
    await run_pipeline()

Initializing database...
Loading embedding model...


modules.json:   0%|          | 0.00/229 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/122 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/526 [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/9.08M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Starting from 0/10000
+15 | TOTAL: 15/10000
+14 | TOTAL: 29/10000
+9 | TOTAL: 38/10000
+17 | TOTAL: 55/10000
+15 | TOTAL: 70/10000
+14 | TOTAL: 84/10000
+14 | TOTAL: 98/10000
+13 | TOTAL: 111/10000
+7 | TOTAL: 118/10000
+16 | TOTAL: 134/10000
+9 | TOTAL: 143/10000
+13 | TOTAL: 156/10000
+14 | TOTAL: 170/10000
+12 | TOTAL: 182/10000
+15 | TOTAL: 197/10000
+15 | TOTAL: 212/10000
+16 | TOTAL: 228/10000
+15 | TOTAL: 243/10000
+16 | TOTAL: 259/10000
+15 | TOTAL: 274/10000
+16 | TOTAL: 290/10000
+8 | TOTAL: 298/10000
+14 | TOTAL: 312/10000
+14 | TOTAL: 326/10000
+8 | TOTAL: 334/10000
+19 | TOTAL: 353/10000
+11 | TOTAL: 364/10000
+20 | TOTAL: 382/10000
+14 | TOTAL: 396/10000
+8 | TOTAL: 404/10000
+15 | TOTAL: 419/10000
+11 | TOTAL: 430/10000
+13 | TOTAL: 443/10000
+14 | TOTAL: 457/10000
+14 | TOTAL: 471/10000
+15 | TOTAL: 486/10000
+15 | TOTAL: 501/10000
+14 | TOTAL: 515/10000
+14 | TOTAL: 529/10000
+16 | TOTAL: 545/10000
+16 | TOTAL: 561/10000
+8 | TOTAL: 569/10000
+13 | TOTAL: 582/10000
+12